In [6]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.datasets import load_diabetes

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [7]:
data = load_diabetes()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [8]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Development:", X_dev.shape)
print("Test:", X_test.shape)

Development: (353, 10)
Test: (89, 10)


In [10]:
os.makedirs("test_data", exist_ok=True)

X_test.to_csv(
    "test_data/X_test.csv",
    index=False
)

y_test.to_csv(
    "test_data/y_test.csv",
    index=False
)

print("Test data saved.")

Test data saved.


In [11]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
fold_results = []

for fold, (train_index, val_index) in enumerate(
    kf.split(X_dev),
    start=1
):

    # 1. Current fold ka train aur validation data
    X_train_fold = X_dev.iloc[train_index]
    X_val_fold = X_dev.iloc[val_index]

    y_train_fold = y_dev.iloc[train_index]
    y_val_fold = y_dev.iloc[val_index]

    # 2. Standardization
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)

    # 3. Linear Regression
    model = LinearRegression()

    model.fit(
        X_train_scaled,
        y_train_fold
    )

    # 4. Validation prediction
    y_val_pred = model.predict(
        X_val_scaled
    )

    # 5. Metrics
    rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            y_val_pred
        )
    )

    mae = mean_absolute_error(
        y_val_fold,
        y_val_pred
    )

    r2 = r2_score(
        y_val_fold,
        y_val_pred
    )

    fold_results.append({
        "Fold": fold,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

cv_results = pd.DataFrame(fold_results)

display(cv_results)

,Fold,RMSE,MAE,R2
0,1,53.373689,44.590926,0.470125
1,2,56.760999,48.277649,0.536815
2,3,53.043118,42.926027,0.411083
3,4,54.460037,44.301147,0.491289
4,5,59.335324,45.145249,0.492511


In [13]:
print("Mean RMSE:", cv_results["RMSE"].mean())
print("Std RMSE :", cv_results["RMSE"].std())
print("Mean MAE :", cv_results["MAE"].mean())
print("Mean R2  :", cv_results["R2"].mean())

Mean RMSE: 55.394633328018564
Std RMSE : 2.6402325000024476
Mean MAE : 45.04819964235477
Mean R2  : 0.4803645434411371


In [14]:
final_scaler = StandardScaler()

X_dev_scaled = final_scaler.fit_transform(
    X_dev
)

print("Final scaling completed.")

Final scaling completed.


In [15]:
final_model = LinearRegression()

final_model.fit(
    X_dev_scaled,
    y_dev
)

print("Final Linear Regression model trained.")

Final Linear Regression model trained.


In [16]:
X_test_final = pd.read_csv(
    "saved_test_data/X_test.csv"
)

y_test_final = pd.read_csv(
    "saved_test_data/y_test.csv"
).squeeze()

print("X_test:", X_test_final.shape)
print("y_test:", y_test_final.shape)

X_test: (89, 10)
y_test: (89,)


In [17]:
X_test_scaled = final_scaler.transform(
    X_test_final
)

In [18]:
y_pred = final_model.predict(
    X_test_scaled
)

print("First 10 predictions:")
print(y_pred[:10])

First 10 predictions:
[139.5475584  179.51720835 134.03875572 291.41702925 123.78965872
  92.1723465  258.23238899 181.33732057  90.22411311 108.63375858]


In [19]:
final_rmse = np.sqrt(
    mean_squared_error(
        y_test_final,
        y_pred
    )
)

final_mae = mean_absolute_error(
    y_test_final,
    y_pred
)

final_r2 = r2_score(
    y_test_final,
    y_pred
)

print("================================")
print("       FINAL TEST RESULTS")
print("================================")

print("RMSE:", final_rmse)
print("MAE :", final_mae)
print("R²  :", final_r2)

       FINAL TEST RESULTS
RMSE: 53.853445836765935
MAE : 42.794094679599944
R²  : 0.45260276297191926
